In [1]:
import pandas as pd
import matplotlib.pyplot as plt

traffic = pd.read_csv("../data/raw/traffic.csv")
air = pd.read_csv("../data/raw/air_quality.csv")
weather = pd.read_csv("../data/raw/weather.csv")
energy = pd.read_csv("../data/raw/energy.csv")

print("Traffic shape:", traffic.shape)
print("Air quality shape:", air.shape)
print("Weather shape:", weather.shape)
print("Energy shape:", energy.shape)

traffic.head()


Traffic shape: (48204, 9)
Air quality shape: (43824, 13)
Weather shape: (3650, 2)
Energy shape: (4383, 5)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [2]:
air.columns


Index(['No', 'year', 'month', 'day', 'hour', 'pm2.5', 'DEWP', 'TEMP', 'PRES',
       'cbwd', 'Iws', 'Is', 'Ir'],
      dtype='object')

In [3]:
import pandas as pd

# --- Traffic ---
traffic['date_time'] = pd.to_datetime(traffic['date_time'])

# --- Air Quality ---
# Create a proper datetime from the separate columns
air['datetime'] = pd.to_datetime(
    air[['year', 'month', 'day', 'hour']]
)

# --- Weather ---
# The column is likely named 'Date' already, but this handles either case
if 'Date' in weather.columns:
    weather['Date'] = pd.to_datetime(weather['Date'])
elif 'date' in weather.columns:
    weather.rename(columns={'date': 'Date'}, inplace=True)
    weather['Date'] = pd.to_datetime(weather['Date'])

# --- Energy ---
energy['Date'] = pd.to_datetime(energy['Date'])

# Sort each dataset by its time column
traffic = traffic.sort_values('date_time')
air = air.sort_values('datetime')
weather = weather.sort_values('Date')
energy = energy.sort_values('Date')

print("Datetime columns created and datasets sorted.")


Datetime columns created and datasets sorted.


In [4]:
# create a date-only column for joining daily weather with hourly traffic
traffic['date_only'] = traffic['date_time'].dt.date
weather['date_only'] = weather['Date'].dt.date

# merge traffic with weather on date
traffic_weather = traffic.merge(weather, on='date_only', how='left')

print("Merged shape:", traffic_weather.shape)
traffic_weather.head()


Merged shape: (48204, 12)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,date_only,Date,Temp
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,2012-10-02,NaT,NaN
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,2012-10-02,NaT,NaN
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,2012-10-02,NaT,NaN
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,2012-10-02,NaT,NaN
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,2012-10-02,NaT,NaN


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Work on a copy
t = traffic.copy()

# 1) Basic cleaning
# temp is in Kelvin -> convert to Celsius
t['temp_c'] = t['temp'] - 273.15

# fill holiday as a category (some rows are NaN)
t['holiday'] = t['holiday'].fillna('None')

# 2) Time features
t['hour'] = t['date_time'].dt.hour
t['dow'] = t['date_time'].dt.dayofweek   # 0=Mon
t['month'] = t['date_time'].dt.month
t['is_weekend'] = (t['dow'] >= 5).astype(int)

# 3) Encode weather_main (small cardinality)
t['weather_main'] = t['weather_main'].astype('category')
t['weather_main_code'] = t['weather_main'].cat.codes

# 4) Feature set / target
features = [
    'temp_c','rain_1h','snow_1h','clouds_all',
    'hour','dow','month','is_weekend',
    'weather_main_code'
]
X = t[features]
y = t['traffic_volume']

# 5) Chronological split (first 80% train, last 20% test)
t_sorted = t.sort_values('date_time')
split_idx = int(len(t_sorted)*0.8)
X_train, X_test = t_sorted[features].iloc[:split_idx], t_sorted[features].iloc[split_idx:]
y_train, y_test = t_sorted['traffic_volume'].iloc[:split_idx], t_sorted['traffic_volume'].iloc[split_idx:]

# 6) Train baseline model
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# 7) Evaluate
pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"MAE: {mae:,.1f}")
print(f"R^2: {r2:.3f}")

# 8) Quick feature importance
fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
fi


MAE: 308.5
R^2: 0.924


hour                 0.829197
dow                  0.061958
is_weekend           0.048285
temp_c               0.034779
month                0.010805
clouds_all           0.007221
weather_main_code    0.004628
rain_1h              0.002954
snow_1h              0.000174
dtype: float64

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

t = traffic.copy()
t['temp_c'] = t['temp'] - 273.15
t['hour'] = t['date_time'].dt.hour
t['dow'] = t['date_time'].dt.dayofweek
t['month'] = t['date_time'].dt.month

X = t[['temp_c','hour','dow','month','clouds_all','rain_1h','snow_1h']]
y = t['traffic_volume']

split = int(len(t)*0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R²:", r2_score(y_test, pred))


MAE: 313.62986094111955
R²: 0.9202854555538945


In [8]:
import os
os.makedirs("../models", exist_ok=True)


In [9]:
import joblib
joblib.dump(rf, "../models/traffic_rf.pkl")


['../models/traffic_rf.pkl']

In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import os

# ensure models folder exists (relative to notebook in notebooks/)
os.makedirs("../models", exist_ok=True)

# 1) Load air quality (pollution.csv shaped)
air = pd.read_csv("../data/raw/air_quality.csv")

# Common columns for this dataset:
# ['No','year','month','day','hour','pm2.5','DEWP','TEMP','PRES','cbwd','Iws','Is','Ir']
# Build datetime
air['datetime'] = pd.to_datetime(air[['year','month','day','hour']])
air = air.sort_values('datetime').reset_index(drop=True)

# 2) Basic cleaning
# pm2.5 may have NaNs at start/end—drop them safely
air = air.dropna(subset=['pm2.5','TEMP','DEWP','PRES','Iws','Is','Ir'])

# 3) Create time features & a 1-hour lag of pm2.5
air['hour'] = air['datetime'].dt.hour
air['dow']  = air['datetime'].dt.dayofweek
air['month']= air['datetime'].dt.month
air['pm25_lag1'] = air['pm2.5'].shift(1)

# Drop the first row where lag is NaN
air = air.dropna(subset=['pm25_lag1']).reset_index(drop=True)

# 4) Define features/target for next-hour pm2.5
features_air = ['TEMP','DEWP','PRES','Iws','Is','Ir','hour','dow','month','pm25_lag1']
# Target is pm2.5 one step AHEAD (next hour)
air['pm25_next'] = air['pm2.5'].shift(-1)
air = air.dropna(subset=['pm25_next']).reset_index(drop=True)

X_air = air[features_air]
y_air = air['pm25_next']

# 5) Chronological split
split_air = int(len(air)*0.8)
X_air_train, X_air_test = X_air.iloc[:split_air], X_air.iloc[split_air:]
y_air_train, y_air_test = y_air.iloc[:split_air], y_air.iloc[split_air:]

# 6) Train
air_rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
air_rf.fit(X_air_train, y_air_train)

# 7) Evaluate
air_pred = air_rf.predict(X_air_test)
air_mae = mean_absolute_error(y_air_test, air_pred)
air_r2  = r2_score(y_air_test, air_pred)
print(f"AIR - MAE: {air_mae:.2f}, R²: {air_r2:.3f}")

# 8) Save model + metadata (feature order)
joblib.dump(
    {"model": air_rf, "features": features_air},
    "../models/air_rf.pkl"
)
print("Saved: ../models/air_rf.pkl")


AIR - MAE: 19.56, R²: 0.881
Saved: ../models/air_rf.pkl


In [12]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib, os

# ensure models/ exists (relative to notebook in notebooks/)
os.makedirs("../models", exist_ok=True)

# 1) Load energy dataset you saved earlier (OPSd Germany daily)
energy = pd.read_csv("../data/raw/energy.csv")

# Handle common column name variants robustly
cols = {c.lower(): c for c in energy.columns}
date_col = cols.get('date', 'Date')
consumption_col = None
for key in ['consumption','load','demand','total_load']:
    if key in cols:
        consumption_col = cols[key]
        break
if consumption_col is None:
    # OPSD file usually has "Consumption"
    consumption_col = 'Consumption'

# 2) Parse datetime and sort
energy[date_col] = pd.to_datetime(energy[date_col])
energy = energy.sort_values(date_col).reset_index(drop=True)

# 3) Keep just date + consumption, drop NaNs
df = energy[[date_col, consumption_col]].rename(columns={date_col:'Date', consumption_col:'Consumption'})
df = df.dropna(subset=['Consumption']).reset_index(drop=True)

# 4) Create features for next-day forecast
df['lag1']  = df['Consumption'].shift(1)
df['lag7']  = df['Consumption'].shift(7)
df['roll7'] = df['Consumption'].rolling(7).mean()
df['roll30']= df['Consumption'].rolling(30).mean()
df['dow']   = df['Date'].dt.dayofweek
df['month'] = df['Date'].dt.month

# target = next day's consumption
df['y_next'] = df['Consumption'].shift(-1)

# Drop rows with missing lags/rolling/target
df = df.dropna(subset=['lag1','lag7','roll7','roll30','y_next']).reset_index(drop=True)

features_energy = ['lag1','lag7','roll7','roll30','dow','month']
X_en = df[features_energy]
y_en = df['y_next']

# 5) Chronological split
split_en = int(len(df)*0.8)
X_en_train, X_en_test = X_en.iloc[:split_en], X_en.iloc[split_en:]
y_en_train, y_en_test = y_en.iloc[:split_en], y_en.iloc[split_en:]

# 6) Train
en_rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
en_rf.fit(X_en_train, y_en_train)

# 7) Evaluate
en_pred = en_rf.predict(X_en_test)
en_mae  = mean_absolute_error(y_en_test, en_pred)
en_r2   = r2_score(y_en_test, en_pred)
print(f"ENERGY - MAE: {en_mae:.2f}, R²: {en_r2:.3f}")

# 8) Save model + required feature order
joblib.dump({"model": en_rf, "features": features_energy}, "../models/energy_rf.pkl")
print("Saved: ../models/energy_rf.pkl")


ENERGY - MAE: 30.94, R²: 0.900
Saved: ../models/energy_rf.pkl
